# Полные предложения с подвохами на больших моделях

Тот же прогон, что `scripts/sentences_tricks.py` (протокол записан в начале файла и закоммичен до прогона), на моделях, которые на ноутбуке считать долго: GPT-2 large, Qwen2.5-1.5B, Qwen2.5-3B.

Интерпретатор здесь не обучается: его веса лежат в архиве (`results/sentences_tricks_interpreter.pt`), поэтому большие модели читаются теми же весами, что и в основном прогоне.

**Перед запуском:** меню «Среда выполнения» → «Сменить среду выполнения» → GPU (T4). Затем запускайте ячейки по очереди.

In [ ]:
!nvidia-smi

Код скачивается прямо из репозитория, вместе с весами интерпретатора.

In [ ]:
!rm -rf whybox && git clone -q https://github.com/knbww/whybox.git
%cd whybox

In [ ]:
!pip install -q "transformers>=4.56" pyyaml pytest

Проверка, что предложения грамматичны до и после каждого подавления. Тест токенов пропускается, если токенизаторы ещё не скачаны.

In [ ]:
!python -m pytest -q tests/test_tricks.py

## GPT-2 large и Qwen2.5-1.5B

float32, на T4 около 15–20 минут.

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/sentences_tricks.py \
    --models gpt2-large Qwen/Qwen2.5-1.5B-Instruct \
    --device cuda --dtype float32 \
    --out results/sentences_tricks_colab.json

## Qwen2.5-3B

В float32 вместе с градиентами в T4 не помещается, поэтому float16. Хватает ли точности, покажет ошибка полноты в выводе: она должна быть того же порядка, что у моделей выше.

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/sentences_tricks.py \
    --models Qwen/Qwen2.5-3B-Instruct \
    --device cuda --dtype float16 \
    --out results/sentences_tricks_colab_3b.json

## Итог и скачивание результатов

In [ ]:
from google.colab import files
import glob, json
for f in sorted(glob.glob('results/sentences_tricks_colab*.json')):
    d = json.load(open(f))
    for model, by_t in d['results'].items():
        for t, r in by_t.items():
            acc = {k: round(v, 3) for k, v in r['arms']['interpreter'].items() if v == v}
            print(f"{model:30s} {t:8s} {r['read_out']:26s} interpreter {acc}  "
                  f"live {r['arms']['live']['ALL']:.3f}  completeness {r['completeness_residual'][0]:.4f}")
for f in glob.glob('results/sentences_tricks_colab*.json'):
    files.download(f)